# P9: char-darajali LSTM bilan matn generatsiya
**Mavzu:** char-LSTM generativ model, temperature sampling, Bi-LSTM va perplexity
**Kun:** 10-kun amaliyoti — 30-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L9 — Matn generatsiya va ikki tomonlama RNN (`d09_matn_generatsiya.pdf`)
**Kapstone modul:** m09 `TextGenerator` (pedagogik)
**Ma'lumot:** uz_news_full / badiiy parcha. Offline: `d10_checkpoints/uz_sheriy_korpus.txt`.

## Bugungi maqsadlar (ma'ruza [C] dan)
1. Temperature softmax ni ($T{=}1$ da $p(\text{nlp}){=}0.665$) qo'lda hisoblaymiz.
2. char-darajali LSTM generativ modelni o'qitamiz va generatsiya qilamiz.
3. Temperature 0.3/0.7/1.2 da natijalarni taqqoslaymiz.
4. Bi-LSTM va perplexity orqali tushunishni baholaymiz (generatsiya emas).

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | seedlar, OFFLINE_FALLBACK, HAS_TORCH |
| §2 Yaxlit natija | 8 daq | tayyor generate demo |
| §3 Periferiya (PRIMM) | 17 daq | char korpus + char tokenizatsiya; next-char loop |
| §4 Yadro | 35 daq | temperature softmax (qulflangan), char-LSTM train/generate, temperature, perplexity |
| §5 Loyihaga ulash | 13 daq | m09 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | temperature naqshi; nega Bi-LSTM generatsiya qilmaydi |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq, `torch` bayrog'i. GPU shart emas (kichik char-LSTM CPU da).

In [ ]:
# Kaggle: torch oldindan o'rnatilgan (GPU bo'lsa tezlashadi, talab emas)
import os, sys, random, math
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True

try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
    print("torch mavjud:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch yo'q — char n-gram + temperature ishlatiladi (offline rejim).")

MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")
sys.path.insert(0, MODULES)
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Tayyor `TextGenerator` char-darajali korpusda o'qitiladi va seed dan matn davom ettiradi.
Tafsilotlarni keyin ochamiz. (Kichik korpus + kam epoch $\Rightarrow$ natija sodda.)

In [ ]:
from m09_text_generator import TextGenerator

CORPUS = os.path.join("..", "..", "course", "practices", "d10_checkpoints", "uz_sheriy_korpus.txt")
if not os.path.exists(CORPUS):
    CORPUS = os.path.join("course", "practices", "d10_checkpoints", "uz_sheriy_korpus.txt")

text = open(CORPUS, encoding="utf-8").read()
print("Korpus belgilar soni:", len(text), "| char vocab:", len(set(text)))

gen_demo = TextGenerator()
gen_demo.train(text, epochs=12, hidden_size=64)
print("Generatsiya (T=0.7):")
print(gen_demo.generate("tong ", length=60, temperature=0.7))


## 3. Tayyor kod bloki — char tokenizatsiya va next-char loop (PRIMM)
Bu bo'lim periferiya. char-darajali modelda har **belgi** birlik. Matnni belgi$\leftrightarrow$indeks
ga o'giramiz va next-char (keyingi belgi) bashorat juftlarini quramiz.

**Bashorat qiling:** "tong" so'zi nechta belgiga ajraladi? char vocab'ga bo'sh joy (` `) kiradimi?

In [ ]:
# PERIFERIYA — to'liq beriladi. char tokenizatsiya va next-char juftlari.
chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}

def encode(s):
    return [c2i[ch] for ch in s if ch in c2i]

# next-char juftlari (input belgi -> keyingi belgi)
seq = encode(text)
pairs = [(seq[i], seq[i + 1]) for i in range(len(seq) - 1)]
print("char vocab hajmi:", len(chars))
print("'tong' kodlangan:", encode("tong"))
print("next-char juftlar soni:", len(pairs))
print("Birinchi juft:", (i2c[pairs[0][0]], i2c[pairs[0][1]]))


**Tekshiring:**
1. char vocab'da bo'sh joy va `\n` bormi? Nega ular ham belgi?
2. Nega char-darajali model kichik vocab (≈30) bilan ishlaydi, so'z-darajali esa minglab?
3. next-char bashorat — bu autoregressiv generatsiyaning asosi (L9 [G1]).

**O'zgartiring:** `text` o'rniga o'z qisqa matningizni bering va char vocab'ni ko'ring.

In [ ]:
# PERIFERIYA — next-char training loop sxemasi (illyustrativ; torch-ixtiyoriy).
if HAS_TORCH:
    import torch.nn as nn
    def train_step(model, X, Y, opt, loss_fn):
        opt.zero_grad()
        logits, _ = model(X)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # L9 [I3] clipping
        opt.step()
        return float(loss)
    print("Training step tayyor (gradient clipping bilan, L9 [I3]).")
else:
    print("Offline: char n-gram (sanashga asoslangan, training loop shart emas).")


### Checkpoint A
Orqada qolsangiz — korpus va char vocab'ni qaytadan yuklaydi.

In [ ]:
if OFFLINE_FALLBACK or "text" not in dir():
    text = open(CORPUS, encoding="utf-8").read()
    chars = sorted(set(text))
print("Checkpoint: korpus tayyor —", len(text), "belgi,", len(chars), "char vocab")


## 4. Asosiy mavzu — generatsiya (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning temperature misolini takrorlaymiz.

### 4A. Namuna (men qilaman): temperature softmax
Ma'ruza L9 [I2]: logitlar $z=[3,1,2]$, vocab $=$ [nlp, foydali, qiziq]. $T{=}1$ oddiy softmax.
Toza-numpy (torch shart emas).

In [ ]:
# Temperature softmax (toza-numpy) — L9 [I2]
z = np.array([3.0, 1.0, 2.0])      # vocab: nlp, foydali, qiziq
p_T1 = np.exp(z) / np.exp(z).sum()
p_nlp_T1 = p_T1[0]
print("T=1 p(nlp) =", round(p_nlp_T1, 4))

assert abs(p_nlp_T1 - 0.665) < 1e-3   # Ma'ruza L9 [I2]-slayd bilan solishtiring
print("To'g'ri! T=1 da p(nlp) = 0.665")


### 4B. Birgalikda (biz qilamiz): char-LSTM o'qitamiz va generatsiya qilamiz
`TextGenerator` ichida char-LSTM (Kaggle) yoki char n-gram (offline). CPU uchun kichik
o'lcham va kam epoch (to'liq miqyos: hidden=128, 2 qatlam, epochs=20).

In [ ]:
gen = TextGenerator()
# Char-darajali modelni o'qiting: epochs=12, hidden_size=64
gen.train(text, epochs=12, hidden_size=64)
out = gen.generate("tong ", length=50, temperature=0.7)
print(out)


In [ ]:
assert len(gen._chars) > 0, "char vocab bo'sh — train() chaqirilganini tekshiring."
assert isinstance(out, str) and len(out) >= 50, "generate() so'ralgan uzunlikdagi str qaytarishi kerak."
assert all(c in set(gen._chars) for c in out), "Barcha belgilar korpus vocab'idan bo'lishi kerak."
print("Ajoyib! Model o'qidi va", len(out), "belgili matn generatsiya qildi.")


### 4C. Birgalikda (biz qilamiz): temperature ta'sirini taqqoslaymiz
Past $T$ (0.3) — deterministik, takrorlanuvchi; yuqori $T$ (1.2) — ijodiy, tasodifiy (L9 [I2]).

In [ ]:
natijalar = {T: gen.generate("tong ", length=40, temperature=T) for T in (0.3, 0.7, 1.2)}
for T, s in natijalar.items():
    print(f"T={T}: {s}")


In [ ]:
assert set(natijalar.keys()) == {0.3, 0.7, 1.2}, "Uchta temperature bo'lishi kerak."
assert all(isinstance(s, str) and len(s) >= 40 for s in natijalar.values()), \
    "Har generatsiya so'ralgan uzunlikdagi str bo'lishi kerak."
print("To'g'ri! Uchala temperature da generatsiya ishladi.")


### 4D. Mustaqil (siz qilasiz): perplexity va Bi-LSTM nuansi
Char-bigram perplexity hisoblang (model matnga qanchalik "hayron"). So'ng eslang:
**Bi-LSTM generatsiya qila olmaydi** (L9 — kelajak kerak), u faqat **tushunish** uchun.
Tayanch yo'q.

In [ ]:
from collections import Counter

# char-bigram perplexity (add-1 smoothing) — model sifatini baholash
V = len(chars)
bi = Counter((text[i], text[i + 1]) for i in range(len(text) - 1))
uni = Counter(text)
test = "tong otdi"
log_sum = 0.0
for i in range(len(test) - 1):
    a, b = test[i], test[i + 1]
    p = (bi[(a, b)] + 1) / (uni[a] + V)
    log_sum += math.log(p)
ppl = math.exp(-log_sum / (len(test) - 1))
print("char-bigram perplexity ('tong otdi') =", round(ppl, 2))

# Bi-LSTM: tushunish uchun (generatsiya EMAS) — chiqish o'lchami ikki baravar
if HAS_TORCH:
    import torch.nn as nn
    bilstm = nn.LSTM(16, 32, batch_first=True, bidirectional=True)
    o, _ = bilstm(torch.zeros(1, 5, 16))
    assert o.shape[-1] == 64, "Bidirectional chiqish 2*hidden=64 bo'lishi kerak."
    print("Bi-LSTM chiqish o'lchami:", o.shape[-1], "(2*32) — tushunish uchun, generatsiya emas.")

assert ppl > 1.0 and ppl < V, "Perplexity 1 va vocab hajmi orasida bo'lishi kerak."
print("Mustaqil topshiriq bajarildi: perplexity hisoblandi, Bi-LSTM nuansi tushunildi.")


## 5. Loyihaga ulash — m09 `TextGenerator`
Bugungi ish `capstone/modules/m09_text_generator.py` da (shartnoma `capstone/contracts.py`).
Pedagogik (yakuniy pipelineda emas), char-darajali, torch-ixtiyoriy.

In [ ]:
from m09_text_generator import TextGenerator

g = TextGenerator()
for metod in ["train", "generate"]:
    assert callable(getattr(g, metod, None)), f"m09.{metod} mavjud emas!"
g.train("salom dunyo salom", epochs=3, hidden_size=16)
assert isinstance(g.generate("sa", length=10), str), "generate() str qaytarishi kerak."
print("m09 TextGenerator shartnomaga mos: train / generate (char-darajali)")


```bash
git add capstone/modules/m09_text_generator.py
git commit -m "P9: m09 TextGenerator — char-LSTM generatsiya"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba:** temperature 0.3 va 1.2 generatsiyalarini solishtiring. Qaysi biri ko'proq
\textbf{takrorlanadi}? Qaysi biri ko'proq \textbf{xato} belgilar beradi?

In [ ]:
past = gen.generate("bahor ", length=60, temperature=0.3)
yuqori = gen.generate("bahor ", length=60, temperature=1.2)
print("T=0.3 (deterministik):", past)
print("T=1.2 (ijodiy):", yuqori)
print("Eslatma: past T takrorlanuvchi, yuqori T tasodifiy/xatoroq (L9 [I2]).")


**Savol (o'ylab ko'ring):** o'zbek badiiy matnini generatsiya qilishda qaysi temperature
tabiiyroq? Nega \textbf{bidirectional} LSTM matn generatsiya qila olmaydi (L9)? Qaysi
vazifada (teglash/NER) u foydali?

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_